# EigenVectorsValues

Bu çalışma üç ana bölümden oluşur:
1. Teorik açıklama
2. NumPy `linalg.eig` incelemesi
3. Hazır `eig` kullanmadan özdeger hesabı ve karşılaştırma


## 1. Teorik Arka Plan

### Matris Manipülasyonu
Makine öğrenmesinde veri matris şeklinde tutulur. Satırlar gözlemleri, sütunlar özellikleri temsil eder.
Bu nedenle matris çarpımı, transpoz, ters alma ve determinant gibi lineer cebir işlemleri modelleme sürecinin temelini oluşturur.

### Özdeğer ve Özvektör
Bir kare matris için `A v = λ v` eşitliğini sağlayan `λ` özdeğer, `v` ise özvektördür.
Özvektör, matris dönüşümünden sonra yönünü koruyan yönü; özdeğer ise bu yönün ne kadar ölçeklendiğini gösterir.

### Makine Öğrenmesindeki Yeri
Bu kavramlar özellikle PCA, spektral yöntemler, kovaryans analizi, veri sıkıştırma ve görüntü işleme gibi alanlarda kullanılır.


## 2. NumPy `linalg.eig`

`numpy.linalg.eig(a)` kare bir matrisin özdeğerlerini ve sağ özvektörlerini hesaplar.

Temel noktalar:
- Çıktı iki parçadır: özdeğerler ve özvektörler
- Özvektörler sütunlar halindedir
- Genel kare matrisler için kullanılır
- Altyapıda LAPACK `_geev` rutinleri bulunur


In [ ]:
import numpy as np

A = np.array([[6, 1, -1],
              [0, 7,  0],
              [3, -1, 2]], dtype=float)

eigenvalues, eigenvectors = np.linalg.eig(A)

print("Ozdegerler:")
print(eigenvalues)

print("\nOzvektorler:")
print(eigenvectors)


## 3. Hazır `eig` Kullanmadan Uygulama

Aşağıdaki kod, referans GitHub reposundaki mantığı takip eder:
- `A - λI` kurulur
- Determinant denklem olarak elde edilir
- Karakteristik polinomun kökleri bulunur


In [ ]:
import numpy as np

def get_dimensions(matrix):
    return [len(matrix), len(matrix[0])]

def list_multiply(list1, list2):
    result = [0 for _ in range(len(list1) + len(list2) - 1)]
    for i in range(len(list1)):
        for j in range(len(list2)):
            result[i + j] += list1[i] * list2[j]
    return result

def list_add(list1, list2, sub=1):
    return [i + (sub * j) for i, j in zip(list1, list2)]

def identity_matrix(dimensions):
    matrix = [[0 for _ in range(dimensions[1])] for _ in range(dimensions[0])]
    for i in range(dimensions[0]):
        matrix[i][i] = 1
    return matrix

def characteristic_equation(matrix):
    dimensions = get_dimensions(matrix)
    return [[[a, -b] for a, b in zip(row_a, row_i)]
            for row_a, row_i in zip(matrix, identity_matrix(dimensions))]

def determinant_equation(matrix, excluded=[1, 0]):
    dimensions = get_dimensions(matrix)

    if dimensions == [2, 2]:
        tmp = list_add(
            list_multiply(matrix[0][0], matrix[1][1]),
            list_multiply(matrix[0][1], matrix[1][0]),
            sub=-1
        )
        return list_multiply(tmp, excluded)

    new_matrices = []
    excluded_terms = []
    exclude_row = 0

    for exclude_column in range(dimensions[1]):
        tmp = []
        excluded_terms.append(matrix[exclude_row][exclude_column])

        for row in range(1, dimensions[0]):
            tmp_row = []
            for column in range(dimensions[1]):
                if (row != exclude_row) and (column != exclude_column):
                    tmp_row.append(matrix[row][column])
            tmp.append(tmp_row)

        new_matrices.append(tmp)

    determinant_equations = [
        determinant_equation(new_matrices[j], excluded_terms[j])
        for j in range(len(new_matrices))
    ]

    dt_equation = [sum(i) for i in zip(*determinant_equations)]
    return dt_equation

def find_eigenvalues_without_eig(matrix):
    dt_equation = determinant_equation(characteristic_equation(matrix))
    return np.roots(dt_equation[::-1])


In [ ]:
A_list = [[6, 1, -1],
          [0, 7, 0],
          [3, -1, 2]]

manual_eigenvalues = find_eigenvalues_without_eig(A_list)
numpy_eigenvalues, numpy_eigenvectors = np.linalg.eig(np.array(A_list, dtype=float))

print("Hazir eig olmadan bulunan ozdegerler:")
print(manual_eigenvalues)

print("\nNumPy eig ile bulunan ozdegerler:")
print(numpy_eigenvalues)

print("\nSiralanmis karsilastirma:")
print(np.sort(np.real_if_close(manual_eigenvalues)))
print(np.sort(np.real_if_close(numpy_eigenvalues)))

print("\nAyni mi?")
print(np.allclose(
    np.sort(np.real_if_close(manual_eigenvalues)),
    np.sort(np.real_if_close(numpy_eigenvalues))
))


## 4. Yorum

- Manuel yöntem, karakteristik polinom üzerinden özdeğerleri bulur.
- NumPy `eig`, özdeğerlerle birlikte özvektörleri de verir.
- Aynı matris üzerinde iki yöntem aynı özdeğerleri vermektedir.
- Pratik kullanımda NumPy daha güçlü ve daha hızlıdır.


## 5. Kaynaklar

- https://numpy.org/doc/2.1/reference/generated/numpy.linalg.eig.html
- https://raw.githubusercontent.com/numpy/numpy/v2.1.0/numpy/linalg/_linalg.py
- https://github.com/LucasBN/Eigenvalues-and-Eigenvectors
- https://raw.githubusercontent.com/lucasbn/Eigenvalues-and-Eigenvectors/refs/heads/master/main.py
